In [21]:
import cv2
import face_recognition
import os
import urllib.request 
import numpy as np
from deepface import DeepFace
import time
import glob


# ***FACE_recognition livetime_webcamera***

In [28]:
video_capture = cv2.VideoCapture(0)

if not video_capture.isOpened():
    print("Нет доступа к камере - выход")
    exit()

print("Для выхода нажмите 'q' ,переключив раскладку на eng. ")

while True:
    ret, frame = video_capture.read()


    small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)

    rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)  #перевод из bgr(использует opencv) в rgb (использует face_recognition)

    face_locations = face_recognition.face_locations(rgb_small_frame)

    for top, right, bottom, left in face_locations:
        top *= 4
        right *= 4
        bottom *= 4
        left *= 4

        cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)

    cv2.imshow('Video', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

video_capture.release()
cv2.destroyAllWindows()


Для выхода нажмите 'q' ,переключив раскладку на eng. 


# ***OpenCV livetime_webcamera***

In [29]:
face_cascade_filename = "haarcascade_frontalface_default.xml"
eye_cascade_filename = "haarcascade_eye.xml"

face_cascade_url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/" + face_cascade_filename
eye_cascade_url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/" + eye_cascade_filename

if not os.path.exists(face_cascade_filename):
    print(f"Файл '{face_cascade_filename}' не найден, скачивает")
    urllib.request.urlretrieve(face_cascade_url, face_cascade_filename)
    print("Файл скачан")

if not os.path.exists(eye_cascade_filename):
    print(f"Файл '{eye_cascade_filename}' не найден,скачивает ")
    urllib.request.urlretrieve(eye_cascade_url, eye_cascade_filename)
    print("Файл скачан")
    
face_cascade = cv2.CascadeClassifier(face_cascade_filename)
eye_cascade = cv2.CascadeClassifier(eye_cascade_filename)

if face_cascade.empty() or eye_cascade.empty():
    print("Критическая ошибка: Не удалось загрузить каскады из файлов.")
    print("Удалите .xml файлы и попробуйте запустить скрипт снова для их скачивания.")
    exit()

In [30]:
video_capture = cv2.VideoCapture(0)

if not video_capture.isOpened():
    print("Нет доступа к камере - выход")
    exit()

print("Для выхода нажмите 'q' ,переключив раскладку на eng. ")

while True:
    ret, frame = video_capture.read()
    if not ret:
        break

    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray_frame, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
        roi_gray = gray_frame[y:y+h, x:x+w]
        roi_color = frame[y:y+h, x:x+w]
        eyes = eye_cascade.detectMultiScale(roi_gray)
        for (ex, ey, ew, eh) in eyes:
            cv2.rectangle(roi_color, (ex, ey), (ex+ew, ey+eh), (255, 0, 0), 2)

    cv2.imshow('Face and Eye Detector', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

video_capture.release()
cv2.destroyAllWindows()
print("Программа завершена.")


Для выхода нажмите 'q' ,переключив раскладку на eng. 
Программа завершена.


# ***Face_recognition face_id***

In [ ]:
if not os.path.exists("source.jpg"):
    print("Ошибка: Файл 'source.jpg' не найден.")
    exit()

source_image = face_recognition.load_image_file("source.jpg")
try:
    known_face_encoding = face_recognition.face_encodings(source_image)[0]
    known_face_names = ["Kirill"]
except IndexError:
    print("Ошибка: Не удалось найти лицо на 'source.jpg'.")
    exit()

print("Лицо-эталон успешно загружено и обработано.")


Загрузка эталонного изображения (source.jpg)...
Лицо-эталон успешно загружено и обработано.


In [ ]:

video_capture = cv2.VideoCapture(0)
if not video_capture.isOpened():
    print("Ошибка: Не удалось получить доступ к веб-камере.")
    exit()

print("Веб-камера включена. Нажмите 'q' для выхода.")

while True:
    ret, frame = video_capture.read()
    if not ret:
        break

    small_frame = cv2.resize(frame, (0, 0), fx=0.5, fy=0.5)

    rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

    face_locations = face_recognition.face_locations(rgb_small_frame)
    face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

    for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
        matches = face_recognition.compare_faces([known_face_encoding], face_encoding)
        name = "Unknown"

        if True in matches:
            name = known_face_names[0]
            box_color = (0, 255, 0)  
        else:
            name = "Unknown"
            box_color = (0, 0, 255)  
            

      
        top *= 2
        right *= 2
        bottom *= 2
        left *= 2

        height = bottom - top
        width = right - left
        expand_y = int(0.3 * height)  
        expand_x = int(0.1 * width)   

   
        top = max(0, top - expand_y)
        bottom = min(frame.shape[0], bottom + expand_y)
        left = max(0, left - expand_x)
        right = min(frame.shape[1], right + expand_x)

        cv2.rectangle(frame, (left, top), (right, bottom), box_color, 2)
        cv2.rectangle(frame, (left, bottom - 35), (right, bottom), box_color, cv2.FILLED)
        font = cv2.FONT_HERSHEY_DUPLEX
        cv2.putText(frame, name, (left + 6, bottom - 6), font, 1.0, (255, 255, 255), 1)

    cv2.imshow('Video', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

video_capture.release()
cv2.destroyAllWindows()


Веб-камера включена. Нажмите 'q' для выхода.
Обнаружено неизвестное лицо!
Обнаружено неизвестное лицо!
Обнаружено неизвестное лицо!
Обнаружено неизвестное лицо!
Обнаружено неизвестное лицо!
Обнаружено неизвестное лицо!
Обнаружено неизвестное лицо!
Обнаружено неизвестное лицо!
Обнаружено неизвестное лицо!
Обнаружено неизвестное лицо!
Программа завершена.


# ***openCV face_id***

In [22]:
face_cascade_filename = "haarcascade_frontalface_default.xml"
eye_cascade_filename  = "haarcascade_eye.xml"

base_url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/"

def ensure_file(fname):
    if not os.path.exists(fname):
        print(f"Файл '{fname}' не найден, скачиваю...")
        urllib.request.urlretrieve(base_url + fname, fname)
        print("Файл скачан.")

ensure_file(face_cascade_filename)
ensure_file(eye_cascade_filename)

face_cascade = cv2.CascadeClassifier(face_cascade_filename)
eye_cascade  = cv2.CascadeClassifier(eye_cascade_filename)

if face_cascade.empty() or eye_cascade.empty():
    print("Критическая ошибка: Не удалось загрузить каскады из файлов.")
    print("Удалите .xml файлы и запустите снова, чтобы скачать заново.")
    raise SystemExit(1)

SCALE_FACTOR = 1.2
MIN_NEIGHBORS = 5
MIN_SIZE = (80, 80)
THRESHOLD = 1
FACE_SIZE = (160, 160)

if not os.path.exists("kirill.HEIC"):
    print("Ошибка: Файл 'source.jpg' не найден.")
    raise SystemExit(1)

src_bgr = cv2.imread("source.jpg")
if src_bgr is None:
    print("Ошибка: Невозможно прочитать 'source.jpg'")
    raise SystemExit(1)

src_gray = cv2.cvtColor(src_bgr, cv2.COLOR_BGR2GRAY)
faces = face_cascade.detectMultiScale(
    src_gray, scaleFactor=SCALE_FACTOR, minNeighbors=MIN_NEIGHBORS, minSize=MIN_SIZE
)

if len(faces) == 0:
    print("Ошибка: На 'source.jpg' не найдено лицо.")
    raise SystemExit(1)

x, y, w, h = sorted(faces, key=lambda r: r[2]*r[3], reverse=True)[0]
ref_face = src_gray[y:y+h, x:x+w]
ref_face = cv2.resize(ref_face, FACE_SIZE, interpolation=cv2.INTER_CUBIC)
ref_face = cv2.equalizeHist(ref_face)
ref_vec  = cv2.normalize(ref_face.astype("float32"), None, 0, 1, cv2.NORM_MINMAX).flatten()

known_name = "Kirill"


In [26]:
face_xml = "haarcascade_frontalface_default.xml"
base_url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/"
if not os.path.exists(face_xml):
    urllib.request.urlretrieve(base_url + face_xml, face_xml)
face_cascade = cv2.CascadeClassifier(face_xml)
if face_cascade.empty(): raise SystemExit("Не удалось загрузить каскад")

FACE_SIZE = (160,160)
SCALE_FACTOR = 1.2
MIN_NEIGHBORS = 5
MIN_SIZE = (80,80)
COS_THRESHOLD = 0.25   

clahe = cv2.createCLAHE(2.0, (8,8))

def face_vec_from_bgr(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, SCALE_FACTOR, MIN_NEIGHBORS, minSize=MIN_SIZE)
    if len(faces)==0: return None
    x,y,w,h = sorted(faces, key=lambda r:r[2]*r[3], reverse=True)[0]
    roi = gray[y:y+h, x:x+w]
    roi = cv2.resize(roi, FACE_SIZE, interpolation=cv2.INTER_CUBIC)
    roi = clahe.apply(roi)
    vec = roi.astype("float32").reshape(-1)
    n = np.linalg.norm(vec) + 1e-8
    return vec / n

ref_vectors = []
ref_names   = []
for p in sorted(glob.glob("refs/*.jpg")) + sorted(glob.glob("refs/*.png")):
    img = cv2.imread(p)
    v = face_vec_from_bgr(img)
    if v is not None:
        ref_vectors.append(v)
        ref_names.append("Kirill")
print(f"Эталонов загружено: {len(ref_vectors)}")
if len(ref_vectors)==0:
    raise SystemExit("Нет валидных эталонов в папке refs")

ref_vectors = np.stack(ref_vectors, axis=0)

def cos_distance(a, b):
    return float(1.0 - np.dot(a, b))

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened(): raise SystemExit("Нет доступа к камере")


Эталонов загружено: 1


In [27]:


print("Камера включена. 'q' для выхода.")
while True:
    ok, frame = cap.read()
    if not ok: break

    small = cv2.resize(frame, None, fx=0.5, fy=0.5)
    gray  = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    dets  = face_cascade.detectMultiScale(gray, SCALE_FACTOR, MIN_NEIGHBORS, minSize=MIN_SIZE)

    for (x,y,w,h) in dets:
        x2,y2,w2,h2 = x*2,y*2,w*2,h*2

        ey = int(0.3*h2); ex = int(0.1*w2)
        top    = max(0, y2 - ey)
        bottom = min(frame.shape[0], y2 + h2 + ey)
        left   = max(0, x2 - ex)
        right  = min(frame.shape[1], x2 + w2 + ex)

        roi = frame[top:bottom, left:right]
        name, color = "Unknown", (0,0,255)

        v = face_vec_from_bgr(roi)
        if v is not None:
            dists = [cos_distance(v, r) for r in ref_vectors]
            dmin = min(dists)
            if dmin < COS_THRESHOLD:
                name, color = "Kirill", (0,255,0)


        cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
        cv2.rectangle(frame, (left, bottom-35), (right, bottom), color, cv2.FILLED)
        cv2.putText(frame, name, (left+6, bottom-8), cv2.FONT_HERSHEY_DUPLEX, 0.9, (255,255,255), 1)

    cv2.imshow("OpenCV simple ID (cosine + refs)", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()


Камера включена. 'q' для выхода.
